In [2]:
import numpy as np
from scipy.integrate import solve_ivp
from matplotlib import pyplot as plt
from quadrotor_model import nonlin_quad_dyn
import json

In [ ]:
g = 9.81
m = 1.0

# inertia (kg*m^2)
Jx = 0.005
Jy = 0.005
Jz = 0.009

# geometry
l = 0.25   # arm length from COM to rotor

# rotor coefficients, assuming thrust/drag ~ coefficient * W^2 with W in rad/s
kf = 9.0e-6
km = 1.5e-7

# motor lag
tau_m = 0.05

J = np.diag([Jx, Jy, Jz])

W_eq = np.sqrt(m*g/(4*kf))
us_eq = np.array([W_eq]*4)

a_thrust = 2*kf*W_eq/m
a_roll   = 2*l*kf*W_eq/Jx
a_pitch  = 2*l*kf*W_eq/Jy
a_yaw    = 2*km*W_eq/Jz
a_motor  = 1/tau_m

# Hover states
x_hover = np.concatenate((np.zeros(12), np.array([W_eq] * 4)))

0.04698137929009748 0.04698137929009748


In [4]:
A = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # px_dot = vx
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # py_dot = vy
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # pz_dot = vz

    [0, 0, 0, 0, 0, 0, 0, -g, 0, 0, 0, 0, 0, 0, 0, 0],  # vx_dot = -g*theta
    [0, 0, 0, 0, 0, 0, g, 0, 0, 0, 0, 0, 0, 0, 0, 0],   # vy_dot =  g*phi
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     -a_thrust, -a_thrust, -a_thrust, -a_thrust],        # vz_dot

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],   # phi_dot = p
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],   # theta_dot = q
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],   # psi_dot = r

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     0,  a_roll, 0, -a_roll],                            # p_dot
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     -a_pitch, 0, a_pitch, 0],                           # q_dot
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     a_yaw, -a_yaw, a_yaw, -a_yaw],                      # r_dot

    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     -a_motor, 0, 0, 0],                                 # W1_dot
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     0, -a_motor, 0, 0],                                 # W2_dot
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     0, 0, -a_motor, 0],                                 # W3_dot
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
     0, 0, 0, -a_motor]                                  # W4_dot
], dtype=float)

B = np.array([
    [0, 0, 0, 0],     # px_dot
    [0, 0, 0, 0],     # py_dot
    [0, 0, 0, 0],     # pz_dot

    [0, 0, 0, 0],     # vx_dot
    [0, 0, 0, 0],     # vy_dot
    [0, 0, 0, 0],     # vz_dot

    [0, 0, 0, 0],     # phi_dot
    [0, 0, 0, 0],     # theta_dot
    [0, 0, 0, 0],     # psi_dot

    [0, 0, 0, 0],     # p_dot
    [0, 0, 0, 0],     # q_dot
    [0, 0, 0, 0],     # r_dot

    [a_motor, 0, 0, 0],   # W1_dot
    [0, a_motor, 0, 0],   # W2_dot
    [0, 0, a_motor, 0],   # W3_dot
    [0, 0, 0, a_motor]    # W4_dot
], dtype=float)

C = np.array([
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
])

In [5]:
params = {
    "g": g,
    "m": m,
    "Jx": Jx,
    "Jy": Jy,
    "Jz": Jz,
    "kf": kf,
    "km": km,
    "l": l,
    "tau_m": tau_m,
    "W_eq": float(W_eq)
}

with open("config/plant_params.json", "w") as f:
    json.dump(params, f, indent=2)



In [6]:
np.savez(
    "models/hover_linearization.npz",
    A=A,
    B=B,
    C=C,
    x_hover=x_hover,
    u_hover=us_eq
)